In [0]:
%run ../utils/utils

## Insights do Modelo — Detecção de Anomalias de Vendas

In [0]:
#  Responsabilidade ÚNICA deste notebook: expor os resultados do modelo de
#  forma simples e direta, para consumo do time de negócio e do dashboard
#  no Looker. Não recalcula nada do modelo — só lê o que `modelo.py` já
#  produziu.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import json

CAMINHO_METADADOS = "IA/encouders/metadados.json"

##  Ler os resultados do modelo

In [0]:
#  IMPORTANTE: lendo stg_predicoes_completas (treino+teste combinados), não
#  mais stg_teste_predicoes. Decisão de escopo: os insights/dashboard devem
#  cobrir TODOS os pedidos que o modelo pontuou, não só o conjunto de teste
#  (20%) — mesmo critério usado no dashboard do Isolation Forest da squad3.
#  A validação do modelo em si (MSE, teste de hipótese etc., lidos de
#  `metadados` logo abaixo) continua calculada só no teste, por rigor
#  metodológico — isso não muda, só a base usada para os INSIGHTS de negócio.
#  Mantido o nome `pdf_teste` no restante do notebook para não precisar
#  renomear todas as referências abaixo; a coluna `conjunto` (treino/teste)
#  deixa explícito, linha a linha, a origem de cada pedido.
pdf_teste = ler_delta("IA/encouders", "stg_predicoes_completas", STORAGE_OPTIONS).toPandas()

# Colunas de dinheiro podem chegar como DecimalType no Spark, que vira
# decimal.Decimal no Pandas — isso quebra contas/gráficos que misturam com
# float (TypeError: unsupported operand type(s) for -: 'float' and
# 'decimal.Decimal', como já aconteceu no modelo.py). Convertendo pra float
# aqui, uma vez só, evita o mesmo problema em qualquer cálculo deste notebook.
for _col_dinheiro in ["valor_total", "valor_frete", "desconto_total"]:
    if _col_dinheiro in pdf_teste.columns:
        pdf_teste[_col_dinheiro] = pdf_teste[_col_dinheiro].astype(float)

file_client = container_squad1.get_file_client(CAMINHO_METADADOS)
metadados = json.loads(file_client.download_file().readall().decode("utf-8"))

print(f"Total de pedidos avaliados: {len(pdf_teste)}")
if "conjunto" in pdf_teste.columns:
    print(f"  (dos quais {(pdf_teste['conjunto'] == 'treino').sum()} vieram do treino "
          f"e {(pdf_teste['conjunto'] == 'teste').sum()} do teste do modelo)")
print(f"Anomalias encontradas: {pdf_teste['is_anomaly'].sum()} "
      f"({100 * pdf_teste['is_anomaly'].sum() / len(pdf_teste):.2f}%)")
print(f"\nResumo da avaliação do modelo (calculada em modelo.py):")
print(f"  MSE no teste: {metadados.get('avaliacao_mse_teste', 'N/D'):.4f}")
print(f"  Variância explicada aproximada: {metadados.get('avaliacao_variancia_explicada_aprox', 0):.2%}")
print(f"  Features com diferença estatística significativa: "
      f"{metadados.get('avaliacao_qtd_features_significativas', 'N/D')}")


## Segmentação simples de clientes

In [0]:
#  Segmentação de negócio fácil de explicar, baseada no histórico de compras
#  de cada cliente (já calculado no feature engineering):
#  - **Cliente Novo**: nenhum pedido anterior
#  - **Recorrente — Baixo Valor**: já comprou antes, ticket médio abaixo da mediana
#  - **Recorrente — Alto Valor**: já comprou antes, ticket médio acima da mediana


mediana_ticket = pdf_teste.loc[pdf_teste["qtd_pedidos_anteriores_cliente"] > 0, "ticket_medio_historico_cliente"].median()

def classificar_segmento(row):
    if row["qtd_pedidos_anteriores_cliente"] == 0:
        return "Cliente Novo"
    elif row["ticket_medio_historico_cliente"] >= mediana_ticket:
        return "Recorrente - Alto Valor"
    else:
        return "Recorrente - Baixo Valor"

pdf_teste["segmento_cliente"] = pdf_teste.apply(classificar_segmento, axis=1)

resumo_segmento = (
    pdf_teste.groupby("segmento_cliente")
    .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
    .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
    .reset_index()
)

print("===== ANOMALIA POR SEGMENTO DE CLIENTE =====")
display(resumo_segmento)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(resumo_segmento["segmento_cliente"], resumo_segmento["taxa_anomalia_pct"], color="#C44E52")
ax.set_ylabel("% de anomalia")
ax.set_title("Taxa de Anomalia por Segmento de Cliente")
plt.xticks(rotation=15)
for i, v in enumerate(resumo_segmento["taxa_anomalia_pct"]):
    ax.text(i, v, f"{v}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

##  Impacto financeiro das anomalias

In [0]:
#  Além de SABER quais pedidos são anômalos, o negócio precisa saber
#  QUANTO DINHEIRO isso representa: soma e média de valor_total, frete e
#  desconto nos pedidos anômalos, comparado ao normal — e qual fatia do
#  faturamento está "sob suspeita". Classificado por ANO, para o time de
#  negócio comparar a evolução do impacto financeiro ao longo do tempo.

if "dt_pedido" in pdf_teste.columns:
    pdf_teste["dt_pedido"] = pd.to_datetime(pdf_teste["dt_pedido"], errors="coerce")
    pdf_teste["ano_pedido"] = pdf_teste["dt_pedido"].dt.year
else:
    pdf_teste["ano_pedido"] = None
    print("[Aviso] Coluna 'dt_pedido' não encontrada em stg_teste_predicoes — rode o modelo.py atualizado "
          "(que agora salva dt_pedido) para classificar o impacto financeiro por ano.")

colunas_financeiras = [c for c in ["valor_total", "valor_frete", "desconto_total"] if c in pdf_teste.columns]

resumo_financeiro = (
    pdf_teste
    .assign(grupo=lambda df: df["is_anomaly"].map({True: "Anomalia", False: "Normal"}))
    .groupby(["ano_pedido", "grupo"])
    .agg(
        qtd_pedidos=("id_pedido", "count"),
        **{f"soma_{c}": (c, "sum") for c in colunas_financeiras},
        **{f"media_{c}": (c, "mean") for c in colunas_financeiras},
    )
    .round(2)
    .reset_index()
    .sort_values(["ano_pedido", "grupo"])
)

if "soma_valor_total" in resumo_financeiro.columns:
    total_por_ano = resumo_financeiro.groupby("ano_pedido")["soma_valor_total"].transform("sum")
    resumo_financeiro["pct_do_valor_total_do_ano"] = (
        100 * resumo_financeiro["soma_valor_total"] / total_por_ano
    ).round(2)

print("===== IMPACTO FINANCEIRO POR ANO: ANOMALIA x NORMAL =====")
display(resumo_financeiro)

if "soma_valor_total" in resumo_financeiro.columns:
    anomalias_por_ano = resumo_financeiro[resumo_financeiro["grupo"] == "Anomalia"]
    for _, linha in anomalias_por_ano.iterrows():
        rotulo_ano = int(linha["ano_pedido"]) if pd.notna(linha["ano_pedido"]) else "N/D"
        print(f"{rotulo_ano}: pedidos anômalos somam R$ {linha['soma_valor_total']:,.2f} — "
              f"{linha['pct_do_valor_total_do_ano']}% do valor transacionado naquele ano.")

# Gráfico: soma de valor_total por ano, Normal x Anomalia lado a lado
if "soma_valor_total" in resumo_financeiro.columns and resumo_financeiro["ano_pedido"].notna().any():
    pivot_valor = (
        resumo_financeiro
        .pivot(index="ano_pedido", columns="grupo", values="soma_valor_total")
        .fillna(0)
        .sort_index()
    )
    anos = pivot_valor.index.tolist()
    posicoes = list(range(len(anos)))
    largura = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    if "Normal" in pivot_valor.columns:
        ax.bar([p - largura / 2 for p in posicoes], pivot_valor["Normal"], largura, label="Normal", color="#4C72B0")
    if "Anomalia" in pivot_valor.columns:
        ax.bar([p + largura / 2 for p in posicoes], pivot_valor["Anomalia"], largura, label="Anomalia", color="#C44E52")
    ax.set_xticks(posicoes)
    ax.set_xticklabels([str(int(a)) if pd.notna(a) else "N/D" for a in anos])
    ax.set_xlabel("Ano do pedido")
    ax.set_ylabel("Soma de valor_total (R$)")
    ax.set_title("Impacto Financeiro por Ano: Normal x Anomalia")
    ax.legend()
    plt.tight_layout()
    plt.show()

##  Faixa de valor x dia da semana, concentração por cliente (Pareto) e KPIs



In [0]:
MAPA_DIA_SEMANA_PT = {
    "Monday": "Seg", "Tuesday": "Ter", "Wednesday": "Qua", "Thursday": "Qui",
    "Friday": "Sex", "Saturday": "Sab", "Sunday": "Dom",
}
ORDEM_DIA_SEMANA_PT = ["Dom", "Seg", "Ter", "Qua", "Qui", "Sex", "Sab"]

if "dt_pedido" in pdf_teste.columns and pdf_teste["dt_pedido"].notna().any():
    pdf_teste["dia_semana_nome"] = pdf_teste["dt_pedido"].dt.day_name().map(MAPA_DIA_SEMANA_PT)

    pdf_teste["faixa_valor"] = pd.qcut(
        pdf_teste["valor_total"], 4,
        labels=["Q1 (mais baixo)", "Q2", "Q3", "Q4 (mais alto)"],
        duplicates="drop"
    )

    taxa_faixa_dia = (
        pdf_teste
        .groupby(["faixa_valor", "dia_semana_nome"], observed=True)
        .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
        .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
        .reset_index()
    )
    taxa_faixa_dia["dia_semana_nome"] = pd.Categorical(
        taxa_faixa_dia["dia_semana_nome"], categories=ORDEM_DIA_SEMANA_PT, ordered=True
    )
    taxa_faixa_dia = taxa_faixa_dia.sort_values(["faixa_valor", "dia_semana_nome"])

    print("===== TAXA DE ANOMALIA (%) POR FAIXA DE VALOR E DIA DA SEMANA =====")
    display(taxa_faixa_dia)
    display(taxa_faixa_dia.pivot(index="faixa_valor", columns="dia_semana_nome", values="taxa_anomalia_pct"))

    # Versão LARGA (1 coluna por dia, já na ordem Dom->Sab) — é essa que vai
    # para o Looker. Uma tabela normal (não "Tabela Dinâmica") não consegue
    # espalhar uma dimensão em colunas sozinha; publicando já pivotado aqui,
    # no Looker basta adicionar cada dia como métrica (SUM), na ordem
    # desejada — sem depender da ordenação alfabética do Looker.
    taxa_faixa_dia_wide = (
        taxa_faixa_dia
        .pivot(index="faixa_valor", columns="dia_semana_nome", values="taxa_anomalia_pct")
        .reindex(columns=ORDEM_DIA_SEMANA_PT)
        .reset_index()
    )
    taxa_faixa_dia_wide.columns.name = None
    taxa_faixa_dia_wide["faixa_valor"] = taxa_faixa_dia_wide["faixa_valor"].astype(str)
    display(taxa_faixa_dia_wide)

    # Cast final para string (publicação): Categorical do pandas não é bem
    # suportado por spark.createDataFrame.
    taxa_faixa_dia["faixa_valor"] = taxa_faixa_dia["faixa_valor"].astype(str)
    taxa_faixa_dia["dia_semana_nome"] = taxa_faixa_dia["dia_semana_nome"].astype(str)
else:
    taxa_faixa_dia = pd.DataFrame(columns=["faixa_valor", "dia_semana_nome", "qtd_pedidos", "qtd_anomalias", "taxa_anomalia_pct"])
    taxa_faixa_dia_wide = pd.DataFrame(columns=["faixa_valor"] + ORDEM_DIA_SEMANA_PT)
    print("[Aviso] Sem 'dt_pedido' válido — não é possível montar faixa de valor x dia da semana.")

In [0]:
por_cliente = (
    pdf_teste.groupby("id_cliente")["is_anomaly"]
    .sum()
    .astype(int)
    .sort_values(ascending=False)
    .reset_index(name="qtd_pedidos_anomalos")
)
por_cliente = por_cliente[por_cliente["qtd_pedidos_anomalos"] > 0].reset_index(drop=True)

pct_clientes_para_80pct = None
if len(por_cliente) > 0:
    por_cliente["pct_clientes_acumulado"] = (np.arange(1, len(por_cliente) + 1) / len(por_cliente)) * 100
    por_cliente["pct_anomalias_acumulado"] = (
        por_cliente["qtd_pedidos_anomalos"].cumsum() / por_cliente["qtd_pedidos_anomalos"].sum() * 100
    )
    idx_80 = (por_cliente["pct_anomalias_acumulado"].values >= 80).argmax()
    pct_clientes_para_80pct = round(por_cliente.loc[idx_80, "pct_clientes_acumulado"], 2)

    print(f"\n===== CONCENTRAÇÃO DE ANOMALIAS POR CLIENTE =====")
    print(f"Clientes com ao menos 1 pedido anômalo: {len(por_cliente)}")
    print(f"{pct_clientes_para_80pct}% desses clientes concentram 80% dos pedidos anômalos")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(por_cliente["pct_clientes_acumulado"], por_cliente["pct_anomalias_acumulado"], color="#C44E52", linewidth=2)
    ax.plot([0, 100], [0, 100], linestyle="--", color="gray", linewidth=1, label="distribuição uniforme")
    ax.axhline(80, color="#4C72B0", linestyle=":", linewidth=1)
    ax.axvline(pct_clientes_para_80pct, color="#4C72B0", linestyle=":", linewidth=1)
    ax.set_xlabel("% acumulado de clientes (ordenados por qtd. anomalias)")
    ax.set_ylabel("% acumulado de pedidos anômalos")
    ax.set_title("Concentração de Anomalias por Cliente (Pareto)")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    por_cliente = pd.DataFrame(columns=["id_cliente", "qtd_pedidos_anomalos", "pct_clientes_acumulado", "pct_anomalias_acumulado"])
    print("[Aviso] Nenhum cliente com pedido anômalo — não é possível montar a curva de Pareto.")

In [0]:
qtd_pedidos_avaliados = len(pdf_teste)
qtd_anomalias_kpi = int(pdf_teste["is_anomaly"].sum())
taxa_anomalia_pct_kpi = round(100 * qtd_anomalias_kpi / qtd_pedidos_avaliados, 2)
soma_valor_anomalias_kpi = round(pdf_teste.loc[pdf_teste["is_anomaly"], "valor_total"].sum(), 2)
ticket_medio_normal_kpi = round(pdf_teste.loc[~pdf_teste["is_anomaly"], "valor_total"].mean(), 2)
ticket_medio_anomalia_kpi = round(pdf_teste.loc[pdf_teste["is_anomaly"], "valor_total"].mean(), 2)

texto_concentracao = (
    f"{pct_clientes_para_80pct}% dos clientes explicam 80% dos casos"
    if pct_clientes_para_80pct is not None else "N/D"
)

kpi_geral = pd.DataFrame([{
    "qtd_pedidos_avaliados": qtd_pedidos_avaliados,
    "qtd_anomalias": qtd_anomalias_kpi,
    "taxa_anomalia_pct": taxa_anomalia_pct_kpi,
    "soma_valor_anomalias": soma_valor_anomalias_kpi,
    "ticket_medio_normal": ticket_medio_normal_kpi,
    "ticket_medio_anomalia": ticket_medio_anomalia_kpi,
    "pct_clientes_para_80pct_anomalias": pct_clientes_para_80pct,
    "texto_concentracao_clientes": texto_concentracao,
}])

print("\n===== KPIs GERAIS (uma linha, para scorecards no Looker) =====")
display(kpi_geral)

##  Anomalias x Pedidos Cancelados

In [0]:
#  `status_pedido` nunca foi usado como feature do modelo (para não vazar
#  informação pós-fato) — mas é útil AGORA, para checar se o modelo está
#  identificando pedidos que depois foram cancelados (um sinal de validação
#  de negócio, não do treino).

if "status_pedido" in pdf_teste.columns:
    resumo_status = (
        pdf_teste.groupby("status_pedido")
        .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
        .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
        .reset_index()
        .sort_values("taxa_anomalia_pct", ascending=False)
    )

    print("===== TAXA DE ANOMALIA POR STATUS DO PEDIDO =====")
    display(resumo_status)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(resumo_status["status_pedido"], resumo_status["taxa_anomalia_pct"], color="#DD8452")
    ax.invert_yaxis()  # barh desenha a 1ª linha embaixo; como o df já está ordenado
                       # do maior pro menor, invertemos o eixo pra o maior ficar em cima
    ax.set_xlabel("% de anomalia")
    ax.set_title("Taxa de Anomalia por Status do Pedido")
    for i, v in enumerate(resumo_status["taxa_anomalia_pct"]):
        ax.text(v, i, f" {v}%", va="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

    if "Cancelado" in resumo_status["status_pedido"].values:
        taxa_cancelado = resumo_status.loc[resumo_status["status_pedido"] == "Cancelado", "taxa_anomalia_pct"].iloc[0]
        taxa_geral = round(100 * pdf_teste["is_anomaly"].sum() / len(pdf_teste), 2)
        print(f"\nPedidos cancelados têm taxa de anomalia de {taxa_cancelado}%, "
              f"contra {taxa_geral}% da base geral "
              f"({'ACIMA' if taxa_cancelado > taxa_geral else 'ABAIXO'} da média).")
else:
    print("Coluna 'status_pedido' não encontrada — rode o modelo.py atualizado antes.")

##  Comparação Squad1 x Squad3

In [0]:
#  Squad1 e Squad3 alimentam o mesmo modelo (união feita no feature_engineer.py),
#  mas cada squad pode ter um perfil de vendas diferente. Aqui comparamos a
#  taxa de anomalia e o tipo de anomalia mais comum entre as duas origens —
#  útil para saber se um dos squads está gerando dados "mais estranhos" que
#  o outro (pode indicar problema de qualidade de dados específico daquele
#  squad, e não necessariamente uma anomalia de negócio real).

if "origem_squad" in pdf_teste.columns:
    resumo_squad = (
        pdf_teste.groupby("origem_squad")
        .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
        .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
        .reset_index()
        .sort_values("taxa_anomalia_pct", ascending=False)
    )

    print("===== TAXA DE ANOMALIA POR ORIGEM (SQUAD1 x SQUAD3) =====")
    display(resumo_squad)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(resumo_squad["origem_squad"], resumo_squad["taxa_anomalia_pct"], color=["#4C72B0", "#DD8452"])
    ax.set_ylabel("% de anomalia")
    ax.set_title("Taxa de Anomalia: Squad1 x Squad3")
    for i, v in enumerate(resumo_squad["taxa_anomalia_pct"]):
        ax.text(i, v, f"{v}%", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout()
    plt.show()

    if len(resumo_squad) == 2:
        maior = resumo_squad.iloc[0]
        menor = resumo_squad.iloc[1]
        print(f"\n{maior['origem_squad']} tem taxa de anomalia de {maior['taxa_anomalia_pct']}%, "
              f"contra {menor['taxa_anomalia_pct']}% de {menor['origem_squad']}.")

    # Tipo de anomalia mais comum, separado por squad — mostra se cada squad
    # "quebra" o modelo por um motivo diferente.
    if "feature_dominante" in pdf_teste.columns:
        anomalias_por_squad = pdf_teste[pdf_teste["is_anomaly"]]
        if len(anomalias_por_squad) > 0:
            top_feature_por_squad = (
                anomalias_por_squad.groupby(["origem_squad", "feature_dominante"])
                .size()
                .reset_index(name="qtd")
                .sort_values(["origem_squad", "qtd"], ascending=[True, False])
                .groupby("origem_squad")
                .head(3)
            )
            print("\n===== TOP 3 TIPOS DE ANOMALIA POR ORIGEM =====")
            display(top_feature_por_squad)
else:
    print("Coluna 'origem_squad' não encontrada — rode o feature_engineer.py/modelo.py atualizados antes.")

##  Por que essas anomalias aconteceram — produtos comprados


In [0]:
#  `feature_dominante` diz QUAL FEATURE NUMÉRICA (ex: "razao_frete_valor")
#  mais contribuiu para o erro de reconstrução — mas não diz O QUE a pessoa
#  comprou. Para responder ao cliente/professor "por que esse pedido foi
#  sinalizado" de forma concreta (ex: "porque essa combinação de produtos é
#  rara" — o clássico caso de mercado "cerveja + fralda"), voltamos aos
#  ITENS de cada pedido anômalo e olhamos as categorias envolvidas.
#
#  Ajuste os nomes abaixo se o schema real de `ecommerce_categorias` /
#  `ecommerce_produtos` usar nomes de coluna diferentes.
COLUNA_ID_CATEGORIA = "id_categoria"
COLUNA_NOME_CATEGORIA = "nome_categoria"


def ler_delta_squad3_insights(camada, tabela, storage_opts):
    """Réplica do helper usado em feature_engineer.py — necessária aqui porque
    insights.py roda de forma independente (não importa o notebook de FE)."""
    caminho_final = get_delta_path_squad3(camada, tabela, storage_opts)
    account_name = storage_opts.get("account_name")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")

    return (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .load(caminho_final))


ANALISE_PRODUTOS_DISPONIVEL = False

# Normaliza id_categoria via double -> long -> string: elimina o ".0"
# que aparece quando a coluna de origem é DoubleType/FloatType num dos
# lados (ex: "102.0" em categorias x "110" em produtos, mesmo valor,
# representação diferente) — sem isso o join nunca bate.
def _normalizar_id_categoria(coluna):
    return F.trim(F.col(coluna).cast("double").cast("long").cast("string"))

try:
    # Reconstrói, a partir da Silver bruta do squad3, o mapa
    # id_pedido -> categorias compradas. Sem prefixo (s3_/s1_): desde que o
    # feature_engineer.py passou a usar só squad3, os IDs em pdf_teste
    # também estão sem prefixo. F.trim() nas chaves de junção (sku,
    # id_categoria) evita falhas silenciosas por espaço em branco.
    df_itens_todos = (ler_delta_squad3_insights("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
                       .select("id_pedido", F.trim(F.col("sku")).alias("sku")))

    df_produtos_todos = (ler_delta_squad3_insights("silver", "ecommerce_produtos", STORAGE_OPTIONS)
                          .select(F.trim(F.col("sku")).alias("sku"),
                                  _normalizar_id_categoria(COLUNA_ID_CATEGORIA).alias(COLUNA_ID_CATEGORIA)))

    df_categorias_todos = (ler_delta_squad3_insights("silver", "ecommerce_categorias", STORAGE_OPTIONS)
                            .select(_normalizar_id_categoria(COLUNA_ID_CATEGORIA).alias(COLUNA_ID_CATEGORIA),
                                    COLUNA_NOME_CATEGORIA)
                            .dropDuplicates([COLUNA_ID_CATEGORIA]))

    # Diagnóstico passo a passo: mostra em qual junção os dados se perdem,
    # em vez de só constatar no final que o resultado veio vazio.
    qtd_itens = df_itens_todos.count()
    qtd_produtos = df_produtos_todos.count()
    qtd_categorias_tab = df_categorias_todos.count()

    df_itens_produtos = df_itens_todos.join(df_produtos_todos, "sku", "left")
    qtd_apos_join_produtos = df_itens_produtos.count()
    qtd_categoria_nula = df_itens_produtos.filter(F.col(COLUNA_ID_CATEGORIA).isNull()).count()

    df_pedido_categoria_bruto = df_itens_produtos.join(df_categorias_todos, COLUNA_ID_CATEGORIA, "left")
    qtd_nome_categoria_nula = df_pedido_categoria_bruto.filter(F.col(COLUNA_NOME_CATEGORIA).isNull()).count()

    df_pedido_categoria = (
        df_pedido_categoria_bruto
        .select("id_pedido", COLUNA_NOME_CATEGORIA)
        .dropna(subset=[COLUNA_NOME_CATEGORIA])
        .distinct()
    )

    pdf_pedido_categoria = df_pedido_categoria.toPandas()
    ANALISE_PRODUTOS_DISPONIVEL = len(pdf_pedido_categoria) > 0

    print(f"[Diagnóstico] itens={qtd_itens} | produtos={qtd_produtos} | categorias={qtd_categorias_tab} | "
          f"itens+produtos={qtd_apos_join_produtos} ({qtd_categoria_nula} sem id_categoria) | "
          f"sem nome_categoria após join categorias={qtd_nome_categoria_nula}")

    if not ANALISE_PRODUTOS_DISPONIVEL:
        if qtd_apos_join_produtos > 0 and qtd_categoria_nula == qtd_apos_join_produtos:
            print("[Aviso] TODOS os itens ficaram sem id_categoria após o join por 'sku' — o sku de "
                  "ecommerce_itens_pedido não está batendo com o sku de ecommerce_produtos "
                  "(formato diferente entre as duas tabelas?).")
        elif qtd_apos_join_produtos > 0 and qtd_nome_categoria_nula == qtd_apos_join_produtos:
            amostra_produtos_cat = sorted({r[COLUNA_ID_CATEGORIA] for r in
                                            df_produtos_todos.select(COLUNA_ID_CATEGORIA).distinct().limit(10).collect()})
            amostra_categorias_cat = sorted({r[COLUNA_ID_CATEGORIA] for r in
                                              df_categorias_todos.select(COLUNA_ID_CATEGORIA).distinct().limit(10).collect()})
            print("[Aviso] TODOS os id_categoria ficaram sem nome_categoria — o id_categoria de "
                  "ecommerce_produtos não está batendo com o de ecommerce_categorias "
                  "(tipo/formato diferente entre as duas tabelas?).")
            print(f"  Amostra id_categoria em ecommerce_produtos:   {amostra_produtos_cat}")
            print(f"  Amostra id_categoria em ecommerce_categorias: {amostra_categorias_cat}")
        else:
            print("[Aviso] Mapa pedido->categoria veio vazio — confira os números do diagnóstico acima.")

except Exception as e:
    print(f"[Aviso] Não foi possível montar a análise de produtos: {e}")
    print("Confira se COLUNA_ID_CATEGORIA / COLUNA_NOME_CATEGORIA batem com o "
          "schema real de 'ecommerce_categorias' / 'ecommerce_produtos'.")


###  Gasto médio e categorias sobre-representadas nas anomalias

In [0]:
if ANALISE_PRODUTOS_DISPONIVEL:
    ids_anomalos = set(pdf_teste.loc[pdf_teste["is_anomaly"], "id_pedido"])
    ids_normais = set(pdf_teste.loc[~pdf_teste["is_anomaly"], "id_pedido"])

    ticket_medio_anomalia = pdf_teste.loc[pdf_teste["is_anomaly"], "valor_total"].mean()
    ticket_medio_normal = pdf_teste.loc[~pdf_teste["is_anomaly"], "valor_total"].mean()

    print(f"Ticket médio — Anomalia: R$ {ticket_medio_anomalia:.2f} | Normal: R$ {ticket_medio_normal:.2f} "
          f"({'ACIMA' if ticket_medio_anomalia > ticket_medio_normal else 'ABAIXO'} da média normal)")

    cat_anomalia = pdf_pedido_categoria[pdf_pedido_categoria["id_pedido"].isin(ids_anomalos)]
    cat_normal = pdf_pedido_categoria[pdf_pedido_categoria["id_pedido"].isin(ids_normais)]

    # Contagens ABSOLUTAS (não só %) — necessárias para filtrar no Looker
    # categorias com amostra pequena demais, que geram lift artificialmente
    # alto (ex: 1 pedido anômalo com uma categoria rara vira lift de 10x+
    # só por causa do +0.01 de suavização, sem significar nada de verdade).
    contagem_abs_anomalia = cat_anomalia[COLUNA_NOME_CATEGORIA].value_counts()
    contagem_abs_normal = cat_normal[COLUNA_NOME_CATEGORIA].value_counts()

    freq_anomalia = (contagem_abs_anomalia / max(len(ids_anomalos), 1) * 100).round(1)
    freq_normal = (contagem_abs_normal / max(len(ids_normais), 1) * 100).round(1)

    resumo_categorias = pd.DataFrame({
        "qtd_pedidos_anomalos": contagem_abs_anomalia,
        "qtd_pedidos_normais": contagem_abs_normal,
        "pct_pedidos_anomalos": freq_anomalia,
        "pct_pedidos_normais": freq_normal,
    }).fillna(0.0)
    resumo_categorias["qtd_pedidos_anomalos"] = resumo_categorias["qtd_pedidos_anomalos"].astype(int)
    resumo_categorias["qtd_pedidos_normais"] = resumo_categorias["qtd_pedidos_normais"].astype(int)

    # +0.01 evita divisão por zero sem distorcer o ranking
    resumo_categorias["lift"] = (
        (resumo_categorias["pct_pedidos_anomalos"] + 0.01) /
        (resumo_categorias["pct_pedidos_normais"] + 0.01)
    ).round(2)

    LIMIAR_QTD_MINIMA_CATEGORIA = 5
    resumo_categorias["amostra_suficiente"] = (
        resumo_categorias["qtd_pedidos_anomalos"] >= LIMIAR_QTD_MINIMA_CATEGORIA
    )

    resumo_categorias = resumo_categorias.sort_values("lift", ascending=False)

    print("\n===== CATEGORIAS SOBRE-REPRESENTADAS NAS ANOMALIAS =====")
    print("(lift > 1: a categoria aparece proporcionalmente mais em pedidos anômalos que em normais. "
          f"amostra_suficiente=False quando qtd_pedidos_anomalos < {LIMIAR_QTD_MINIMA_CATEGORIA} — "
          "lift nessas linhas não é confiável, é ruído de amostra pequena.)")
    display(resumo_categorias.head(10))

    top_categorias_plot = resumo_categorias.head(10).sort_values("lift", ascending=True)
    cores_categoria = ["#55A868" if s else "#8C8C8C" for s in top_categorias_plot["amostra_suficiente"]]

    fig, ax = plt.subplots(figsize=(9, 5))
    barras_cat = ax.barh(top_categorias_plot.index, top_categorias_plot["lift"], color=cores_categoria)
    ax.set_xlabel("Lift (quanto maior, mais associada à anomalia)")
    ax.set_title("Categorias Mais Associadas a Pedidos Anômalos")
    for barra, qtd in zip(barras_cat, top_categorias_plot["qtd_pedidos_anomalos"]):
        ax.text(barra.get_width(), barra.get_y() + barra.get_height() / 2, f" {barra.get_width():.1f}x",
                va="center", fontweight="bold", fontsize=9)
    ax.legend(handles=[
        plt.Rectangle((0, 0), 1, 1, color="#55A868", label=f"Amostra ≥ {LIMIAR_QTD_MINIMA_CATEGORIA} pedidos"),
        plt.Rectangle((0, 0), 1, 1, color="#8C8C8C", label=f"Amostra < {LIMIAR_QTD_MINIMA_CATEGORIA} pedidos (pouco confiável)"),
    ], loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()

###  Combinações de categorias nas anomalias (ex: "cerveja + fralda")

In [0]:
#  Regra de associação simplificada (sem lib externa): para cada par de
#  categorias que aparece junto num mesmo pedido anômalo, calcula suporte
#  (% dos pedidos anômalos com as duas juntas), confiança (dado que
#  comprou A, qual % também comprou B) e lift (se a combinação é mais
#  frequente do que se A e B fossem independentes). Lift >> 1 é o sinal
#  clássico de "combinação estranha, mas real" — a mesma lógica por trás
#  do achado de mercado "cerveja + fralda".

if ANALISE_PRODUTOS_DISPONIVEL:
    from itertools import combinations

    cestas_anomalia = cat_anomalia.groupby("id_pedido")[COLUNA_NOME_CATEGORIA].apply(set)
    total_cestas = len(cestas_anomalia)
    contagem_individual = cat_anomalia[COLUNA_NOME_CATEGORIA].value_counts().to_dict()

    contagem_pares = {}
    for cesta in cestas_anomalia:
        for par in combinations(sorted(cesta), 2):
            contagem_pares[par] = contagem_pares.get(par, 0) + 1

    linhas_regras = []
    for (cat_a, cat_b), qtd_junto in contagem_pares.items():
        suporte = qtd_junto / total_cestas
        confianca_a_b = qtd_junto / contagem_individual[cat_a]
        prob_b = contagem_individual[cat_b] / total_cestas
        lift = (confianca_a_b / prob_b) if prob_b > 0 else 0
        linhas_regras.append({
            "categoria_a": cat_a, "categoria_b": cat_b,
            "qtd_pedidos_juntos": qtd_junto,
            "suporte_pct": round(100 * suporte, 2),
            "confianca_pct": round(100 * confianca_a_b, 2),
            "lift": round(lift, 2),
        })

    if linhas_regras:
        df_regras = pd.DataFrame(linhas_regras).sort_values("lift", ascending=False)
        print("===== TOP 10 COMBINAÇÕES DE CATEGORIAS MAIS FORTES NAS ANOMALIAS =====")
        display(df_regras.head(10))

        top_regra = df_regras.iloc[0]
        print(f"\nCombinação mais forte: '{top_regra['categoria_a']}' + '{top_regra['categoria_b']}' "
              f"aparece em {top_regra['qtd_pedidos_juntos']} pedidos anômalos (lift {top_regra['lift']}x) — "
              f"esse é o tipo de resposta concreta para o cliente: 'esse pedido foi sinalizado porque "
              f"combina X e Y de um jeito incomum', análogo ao caso clássico de cerveja + fralda.")

        # Gráfico: combinações que MAIS APARECEM (frequência), não só as de
        # maior lift — com poucas anomalias na base, várias combinações
        # empatam com qtd_pedidos_juntos=1 e lift artificialmente igual
        # (ruído de amostra pequena), então ordenar por frequência real é
        # mais confiável para o gráfico do que ordenar só por lift.
        df_regras_plot = df_regras.copy()
        df_regras_plot["combinacao"] = df_regras_plot["categoria_a"] + " + " + df_regras_plot["categoria_b"]
        df_regras_plot = (
            df_regras_plot.sort_values(["qtd_pedidos_juntos", "lift"], ascending=[False, False])
            .head(10)
            .sort_values(["qtd_pedidos_juntos", "lift"], ascending=[True, True])
        )

        fig, ax = plt.subplots(figsize=(9.5, 5.5))
        barras_reg = ax.barh(df_regras_plot["combinacao"], df_regras_plot["qtd_pedidos_juntos"], color="#C44E52")
        ax.set_xlabel("Qtd. de pedidos anômalos com as duas categorias juntas")
        ax.set_title("Combinações de Categorias Mais Frequentes nas Anomalias")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        for barra, lift_val in zip(barras_reg, df_regras_plot["lift"]):
            ax.text(barra.get_width(), barra.get_y() + barra.get_height() / 2,
                    f" {int(barra.get_width())}x  (lift {lift_val:.1f})", va="center", fontweight="bold", fontsize=9)
        plt.tight_layout()
        plt.show()

        if (df_regras["qtd_pedidos_juntos"] <= 1).mean() > 0.5:
            print("Aviso: a maioria das combinações aparece em só 1 pedido anômalo — sinal de que o volume de "
                  "anomalias na base ainda é pequeno. Os lifts individuais são pouco confiáveis; olhe primeiro "
                  "as combinações com maior qtd_pedidos_juntos.")
    else:
        print("Nenhum pedido anômalo tem 2+ categorias distintas para formar combinações.")


###  Produtos e marcas mais associados a pedidos anômalos

In [0]:
#  Mesma lógica do lift por categoria, só que num nível mais granular:
#  produto individual e marca. Ajuste os nomes abaixo se o schema real
#  de `ecommerce_produtos` usar nomes de coluna diferentes.
COLUNA_NOME_PRODUTO = "nome_produto"
COLUNA_MARCA = "nome_marca"

LIMIAR_QTD_MINIMA_PRODUTO = 5
TOP_N_GRAFICO = 5


def calcular_lift(pdf_dimensao, coluna_nome, ids_anomalos, ids_normais, limiar_qtd_minima):
    """Reaproveita o mesmo cálculo de lift já usado para categoria — genérico
    para qualquer dimensão produto/marca/categoria com id_pedido + coluna_nome."""
    dim_anomalia = pdf_dimensao[pdf_dimensao["id_pedido"].isin(ids_anomalos)]
    dim_normal = pdf_dimensao[pdf_dimensao["id_pedido"].isin(ids_normais)]

    contagem_abs_anomalia = dim_anomalia[coluna_nome].value_counts()
    contagem_abs_normal = dim_normal[coluna_nome].value_counts()

    freq_anomalia = (contagem_abs_anomalia / max(len(ids_anomalos), 1) * 100).round(2)
    freq_normal = (contagem_abs_normal / max(len(ids_normais), 1) * 100).round(2)

    resumo = pd.DataFrame({
        "qtd_pedidos_anomalos": contagem_abs_anomalia,
        "qtd_pedidos_normais": contagem_abs_normal,
        "pct_pedidos_anomalos": freq_anomalia,
        "pct_pedidos_normais": freq_normal,
    }).fillna(0.0)
    resumo["qtd_pedidos_anomalos"] = resumo["qtd_pedidos_anomalos"].astype(int)
    resumo["qtd_pedidos_normais"] = resumo["qtd_pedidos_normais"].astype(int)
    resumo["lift"] = ((resumo["pct_pedidos_anomalos"] + 0.01) / (resumo["pct_pedidos_normais"] + 0.01)).round(2)
    resumo["amostra_suficiente"] = resumo["qtd_pedidos_anomalos"] >= limiar_qtd_minima
    return resumo.sort_values("lift", ascending=False)


def plotar_lift_top_n(resumo, titulo, cor_barra="#4C72B0", top_n=TOP_N_GRAFICO):
    """Gráfico de barras horizontais: top N por lift + 'Outros' agrupando o resto
    — mesmo formato do gráfico 'Produtos/Marcas mais associados' do colega."""
    principais = resumo.head(top_n).copy()
    residual = resumo.iloc[top_n:]

    if len(residual) > 0:
        linha_outros = pd.DataFrame([{
            "qtd_pedidos_anomalos": residual["qtd_pedidos_anomalos"].sum(),
            "lift": residual["lift"].mean(),
        }], index=[f"Outros ({len(residual)})"])
        plot_df = pd.concat([principais[["qtd_pedidos_anomalos", "lift"]], linha_outros])
    else:
        plot_df = principais[["qtd_pedidos_anomalos", "lift"]]

    plot_df = plot_df.sort_values("lift", ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(3, 0.6 * len(plot_df))))
    ax.barh(plot_df.index, plot_df["lift"], color=cor_barra)
    ax.set_xlabel("Lift")
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


PRODUTO_MARCA_DISPONIVEL = False
try:
    if "df_itens_todos" not in globals():
        raise NameError("df_itens_todos não foi criado (a leitura de itens/produtos falhou mais acima)")

    # Recalcula aqui por segurança — evita depender de essas variáveis terem
    # sido criadas na seção de categorias logo acima (que pode ter sido
    # pulada se ANALISE_PRODUTOS_DISPONIVEL for False).
    ids_anomalos = set(pdf_teste.loc[pdf_teste["is_anomaly"], "id_pedido"])
    ids_normais = set(pdf_teste.loc[~pdf_teste["is_anomaly"], "id_pedido"])

    colunas_produto_extra = [c for c in [COLUNA_NOME_PRODUTO, COLUNA_MARCA]
                              if c in ler_delta_squad3_insights("silver", "ecommerce_produtos", STORAGE_OPTIONS).columns]

    if colunas_produto_extra:
        df_produtos_nome_marca = (
            ler_delta_squad3_insights("silver", "ecommerce_produtos", STORAGE_OPTIONS)
            .select(F.trim(F.col("sku")).alias("sku"), *colunas_produto_extra)
        )
        df_pedido_produto_bruto = df_itens_todos.join(df_produtos_nome_marca, "sku", "left")
        pdf_pedido_produto = df_pedido_produto_bruto.select("id_pedido", *colunas_produto_extra).toPandas()
        PRODUTO_MARCA_DISPONIVEL = len(pdf_pedido_produto) > 0
    else:
        print(f"[Aviso] Nenhuma das colunas {[COLUNA_NOME_PRODUTO, COLUNA_MARCA]} existe em "
              f"'ecommerce_produtos' — ajuste COLUNA_NOME_PRODUTO/COLUNA_MARCA para os nomes reais.")

except Exception as e:
    print(f"[Aviso] Não foi possível montar a análise de produto/marca: {e}")

if PRODUTO_MARCA_DISPONIVEL and COLUNA_NOME_PRODUTO in pdf_pedido_produto.columns:
    pdf_prod_validos = pdf_pedido_produto.dropna(subset=[COLUNA_NOME_PRODUTO])
    resumo_produtos = calcular_lift(pdf_prod_validos, COLUNA_NOME_PRODUTO, ids_anomalos, ids_normais, LIMIAR_QTD_MINIMA_PRODUTO)

    print("===== TOP PRODUTOS MAIS ASSOCIADOS A PEDIDOS ANÔMALOS (por lift) =====")
    display(resumo_produtos.head(10))
    plotar_lift_top_n(resumo_produtos, "Produtos mais associados a pedidos anômalos", cor_barra="#1E2D4E")
else:
    resumo_produtos = pd.DataFrame(columns=["qtd_pedidos_anomalos", "qtd_pedidos_normais",
                                             "pct_pedidos_anomalos", "pct_pedidos_normais", "lift", "amostra_suficiente"])
    print(f"[Aviso] Coluna '{COLUNA_NOME_PRODUTO}' indisponível — pulei o gráfico de produtos.")

if PRODUTO_MARCA_DISPONIVEL and COLUNA_MARCA in pdf_pedido_produto.columns:
    pdf_marca_validos = pdf_pedido_produto.dropna(subset=[COLUNA_MARCA])
    resumo_marcas = calcular_lift(pdf_marca_validos, COLUNA_MARCA, ids_anomalos, ids_normais, LIMIAR_QTD_MINIMA_PRODUTO)

    print("\n===== TOP MARCAS MAIS ASSOCIADAS A PEDIDOS ANÔMALOS (por lift) =====")
    display(resumo_marcas.head(10))
    plotar_lift_top_n(resumo_marcas, "Marcas mais associadas a pedidos anômalos", cor_barra="#2A8C7E")
else:
    resumo_marcas = pd.DataFrame(columns=["qtd_pedidos_anomalos", "qtd_pedidos_normais",
                                           "pct_pedidos_anomalos", "pct_pedidos_normais", "lift", "amostra_suficiente"])
    print(f"[Aviso] Coluna '{COLUNA_MARCA}' indisponível — pulei o gráfico de marcas.")

##  Tipos de anomalia mais comuns (o que mais chama atenção nas compras estranhas)

In [0]:
anomalias = pdf_teste[pdf_teste["is_anomaly"]]

if "feature_dominante" in pdf_teste.columns and len(anomalias) > 0:
    contagem_tipos = anomalias["feature_dominante"].value_counts().reset_index()
    contagem_tipos.columns = ["tipo_anomalia", "qtd_anomalias"]
    contagem_tipos["pct"] = (100 * contagem_tipos["qtd_anomalias"] / contagem_tipos["qtd_anomalias"].sum()).round(1)

    print("===== TIPOS DE ANOMALIA MAIS COMUNS =====")
    display(contagem_tipos)

    # Gráfico de pizza com muitas fatias pequenas fica ilegível (rótulos se
    # sobrepõem). Aqui: agrupa fatias residuais (< LIMIAR_OUTROS_PCT) em
    # "Outros" e usa barras horizontais ordenadas — muito mais legível quando
    # há uma feature dominante grande e várias pequenas.
    LIMIAR_OUTROS_PCT = 2.0

    principais = contagem_tipos[contagem_tipos["pct"] >= LIMIAR_OUTROS_PCT].copy()
    residual = contagem_tipos[contagem_tipos["pct"] < LIMIAR_OUTROS_PCT]

    if len(residual) > 0:
        linha_outros = pd.DataFrame([{
            "tipo_anomalia": f"Outros ({len(residual)} features)",
            "qtd_anomalias": residual["qtd_anomalias"].sum(),
            "pct": round(residual["pct"].sum(), 1),
        }])
        contagem_plot = pd.concat([principais, linha_outros], ignore_index=True)
    else:
        contagem_plot = principais

    contagem_plot = contagem_plot.sort_values("qtd_anomalias", ascending=True)

    fig, ax = plt.subplots(figsize=(9, max(4, 0.5 * len(contagem_plot))))
    barras = ax.barh(contagem_plot["tipo_anomalia"], contagem_plot["pct"], color="#4C72B0")
    ax.set_xlabel("% das anomalias")
    ax.set_title("Distribuição dos Tipos de Anomalia (feature dominante)")
    for i, v in enumerate(contagem_plot["pct"]):
        ax.text(v, i, f" {v}%", va="center", fontweight="bold")
    ax.set_xlim(0, contagem_plot["pct"].max() * 1.15)
    plt.tight_layout()
    plt.show()

    if len(residual) > 0:
        print(f"'Outros' agrupa as {len(residual)} features com menos de {LIMIAR_OUTROS_PCT}% cada: "
              f"{', '.join(residual['tipo_anomalia'].tolist())}")


## Exemplos concretos — as anomalias mais fortes

In [0]:

colunas_exemplo = ["id_pedido", "id_cliente", "segmento_cliente", "status_pedido",
                   "valor_total", "hora_do_dia", "feature_dominante", "erro_reconstrucao"]
colunas_exemplo_disp = [c for c in colunas_exemplo if c in pdf_teste.columns]

print("===== TOP 15 ANOMALIAS MAIS FORTES =====")
display(
    pdf_teste[pdf_teste["is_anomaly"]]
    .sort_values("erro_reconstrucao", ascending=False)
    .head(15)[colunas_exemplo_disp]
)



## Salvar tabela final — pronta para o Looker

In [0]:
import pyspark.sql.functions as F

# Score 0-100, mais intuitivo para quem for consumir no dashboard
score_min = pdf_teste["erro_reconstrucao"].min()
score_max = pdf_teste["erro_reconstrucao"].max()
pdf_teste["anomaly_score_0_100"] = (
    100 * (pdf_teste["erro_reconstrucao"] - score_min) / (score_max - score_min)
).round(2)

colunas_looker = ["id_pedido", "id_cliente", "segmento_cliente", "status_pedido", "dt_pedido",
                   "origem_squad", "valor_total", "hora_do_dia", "conjunto",
                   "feature_dominante", "anomaly_score_0_100", "is_anomaly"]
colunas_looker = [c for c in colunas_looker if c in pdf_teste.columns]

df_gold_insights = spark.createDataFrame(pdf_teste[colunas_looker]) \
    .withColumn("data_processamento", F.current_timestamp())

gravar_delta(
    df=df_gold_insights,
    camada="gold",
    tabela="gold_insights_anomalias",
    storage_opts=STORAGE_OPTIONS,
    mode="overwrite",
    particionar=False
)

try:
    escrever_sqlserver_gold(df_spark=df_gold_insights, schema="squad1",
                             tabela="gold_insights_anomalias", modo="overwrite")
    print("[Sucesso] gold_insights_anomalias sincronizada no SQL Server (Looker consome daqui).")
except Exception as e:
    print(f"[Aviso] Não foi possível sincronizar com o SQL Server: {e}")

print(f"\ngold_insights_anomalias gravada: {df_gold_insights.count()} linhas.")


## Publicar tabelas-resumo para o Looker (SQL Server)

In [0]:
#  `gold_insights_anomalias` tem granularidade de PEDIDO — ótima para uma
#  tabela detalhada/drill-down no Looker, mas ruim como fonte direta de
#  gráfico de barra/pizza (o Looker teria que agregar toda vez, e cada
#  gráfico recalcularia a mesma coisa). Por isso publicamos também as
#  tabelas JÁ AGREGADAS calculadas acima — uma por pergunta de negócio —
#  como fontes de dado próprias no Looker. Todas em modo "overwrite":
#  cada execução substitui o snapshot anterior (evita o problema de
#  métricas duplicando por causa de modo append + reprocessamento, que já
#  intercorreu no dashboard de volumetria).

def publicar_gold_looker(pdf: pd.DataFrame, tabela: str, indice_para_coluna: str = None):
    """Grava um DataFrame Pandas já agregado como tabela Gold (Delta) e
    espelha no SQL Server, pronto para virar fonte de dado no Looker."""
    pdf_pub = pdf.reset_index() if indice_para_coluna else pdf.copy()
    if indice_para_coluna and pdf_pub.columns[0] != indice_para_coluna:
        pdf_pub = pdf_pub.rename(columns={pdf_pub.columns[0]: indice_para_coluna})

    df_spark_pub = spark.createDataFrame(pdf_pub).withColumn("data_processamento", F.current_timestamp())

    gravar_delta(df=df_spark_pub, camada="gold", tabela=tabela,
                 storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)
    try:
        escrever_sqlserver_gold(df_spark=df_spark_pub, schema="squad1", tabela=tabela, modo="overwrite")
        print(f"[Sucesso] {tabela}: {df_spark_pub.count()} linhas -> Delta + SQL Server.")
    except Exception as e:
        print(f"[Aviso] {tabela} gravada no Delta, mas falhou no SQL Server: {e}")


publicar_gold_looker(resumo_segmento, "gold_encouder_segmento_cliente")

if len(taxa_faixa_dia) > 0:
    publicar_gold_looker(taxa_faixa_dia, "gold_taxa_anomalia_faixa_valor_dia_encouder")

if len(taxa_faixa_dia_wide) > 0:
    publicar_gold_looker(taxa_faixa_dia_wide, "gold_taxa_anomalia_faixa_valor_dia_wide_encouder")

if len(por_cliente) > 0:
    publicar_gold_looker(por_cliente, "gold_pareto_clientes_encouder")

publicar_gold_looker(kpi_geral, "gold_kpi_geral_encouder")

# Comentada: ainda não usada no Looker. Reative (tire o #) quando adicionar
# o card de impacto financeiro (R$ em anomalias) no dashboard.
# if "resumo_financeiro" in globals():
#     publicar_gold_looker(resumo_financeiro, "gold_resumo_financeiro_anomalias")

if "resumo_status" in globals():
    publicar_gold_looker(resumo_status, "gold_encouder_status_pedido")

# Comentada: ainda não usada no Looker (comparativo Squad1 x Squad3).
# if "resumo_squad" in globals():
#     publicar_gold_looker(resumo_squad, "gold_resumo_squad")

if "contagem_tipos" in globals():
    publicar_gold_looker(contagem_tipos, "gold_encouder_feature_dominante")

if "resumo_categorias" in globals():
    publicar_gold_looker(resumo_categorias, "gold_categorias_anomalias_encouder", indice_para_coluna="categoria")

if len(resumo_produtos) > 0:
    publicar_gold_looker(resumo_produtos, "gold_produtos_anomalias_encouder", indice_para_coluna="produto")

if len(resumo_marcas) > 0:
    publicar_gold_looker(resumo_marcas, "gold_marcas_anomalias_encouder", indice_para_coluna="marca")

# Comentada: só a tabela de categorias (acima) está sendo criada por ora.
# Reative se quiser o card de combinações "cerveja + fralda" no Looker.
# if "df_regras" in globals():
#     publicar_gold_looker(df_regras, "gold_regras_associacao_categorias")

print("\nTabelas-resumo publicadas. No Looker, use cada uma como fonte de dado separada "
      "(não recalcule agregações lá — elas já vêm prontas daqui).")